# Generating Synthetic Conversation Datasets with DeepEval

Notebook 4 generated single-turn `Golden`s. A chatbot needs whole **scenarios** — a situation, a
role for the assistant, and a goal the user is trying to reach — which is what a
**`ConversationalGolden`** holds. On its own a `ConversationalGolden` isn't a conversation yet: we
still need something to *play out* the scenario turn by turn, which is what the
**`ConversationSimulator`** does. This notebook builds both halves and ends by scoring the result
with the same conversational metrics from notebook 3.

## 1. Install packages

We only need three libraries: `deepeval` itself, `google-genai` (so DeepEval's judge can talk to
Gemini), and `groq` (the model we're evaluating). No OpenAI key anywhere in this notebook.

In [ ]:
%pip install -q deepeval google-genai groq

## 2. Add your API keys

Two roles, kept separate the whole way through this course:

- **Groq** — the *model under test*. It free-tier serves `llama-3.3-70b-versatile`. Get a key at
  https://console.groq.com/keys
- **Gemini** — the *judge*. Every DeepEval metric needs an LLM to grade with, and we use Gemini so
  you never need an OpenAI key. Get a free key at https://aistudio.google.com/apikey

Keys are entered with `getpass` so they never get printed or saved into the notebook file.

In [ ]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Keys set.")

## The judge: `GeminiModel`

Every DeepEval metric defaults to OpenAI if you don't tell it otherwise — even a metric that
scores deterministically will still try to build an OpenAI client unless you pass `model=`. We
build **one Gemini judge** here and pass it to every metric in this notebook.

In [ ]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-2.5-flash-lite", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

## 4. Generate conversational goldens from scratch

Each `ConversationalGolden` is a *scenario*, not a transcript: who the user is, what they're
trying to do, and what a successful outcome looks like.

In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ConversationalStylingConfig

conv_styling = ConversationalStylingConfig(
    scenario_context="Developers chatting with a GenAI concepts assistant to understand LLM and "
                      "agentic-AI ideas before using them in their own app.",
    conversational_task="Help the user understand a GenAI concept well enough to apply it.",
    participant_roles="A developer (user) and a GenAI concepts assistant.",
)

synthesizer = Synthesizer(model=judge, max_concurrent=1, conversational_styling_config=conv_styling)
scratch_goldens = synthesizer.generate_conversational_goldens_from_scratch(num_goldens=2)

for g in scratch_goldens:
    print("SCENARIO:", g.scenario)
    print("EXPECTED OUTCOME:", g.expected_outcome)
    print("-" * 80)

## 5. Generate conversational goldens grounded in docs

Same `contexts` idea as notebook 4, but each golden becomes a scenario for a conversation about
that material instead of a single Q&A pair.

In [ ]:
DOCS = [
    "Prompt injection is an attack where malicious instructions hidden in user input or retrieved "
    "content trick the model into ignoring its original instructions.",
    "A vector database stores embeddings and finds the nearest vectors to a query embedding "
    "quickly, usually with approximate nearest-neighbor search.",
]
contexts = [[doc] for doc in DOCS]

context_goldens = synthesizer.generate_conversational_goldens_from_contexts(
    contexts=contexts, max_goldens_per_context=1,
)
golden = context_goldens[0]
print("SCENARIO:", golden.scenario)
print("EXPECTED OUTCOME:", golden.expected_outcome)

## 6. Bonus: simulate a full conversation from a golden

A golden only has a *scenario* — no turns to score yet. `ConversationSimulator` plays a simulated
user against **your** chatbot: it calls your `model_callback` for the assistant's reply, decides
what the simulated user says next, and keeps going until the scenario's goal is reached (or the
turn limit hits). Your callback must return a `Turn`, and can accept `input` and `turns`
(the history so far) as parameters -- the simulator only passes the ones your function declares.

In [ ]:
from groq import Groq
from deepeval.test_case import Turn
from deepeval.simulator import ConversationSimulator

groq_client = Groq()
ASSISTANT_SYSTEM = "You are a helpful GenAI concepts assistant. Be concise (1-2 sentences)."


def genai_assistant_callback(input: str, turns: list[Turn]) -> Turn:
    history = [{"role": t.role, "content": t.content} for t in turns]
    messages = [{"role": "system", "content": ASSISTANT_SYSTEM}, *history, {"role": "user", "content": input}]
    reply = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile", messages=messages, temperature=0,
    ).choices[0].message.content
    return Turn(role="assistant", content=reply.strip())


simulator = ConversationSimulator(model_callback=genai_assistant_callback, simulator_model=judge)
simulated_test_cases = simulator.simulate(conversational_goldens=[golden], max_user_simulations=3)

simulated = simulated_test_cases[0]
for t in simulated.turns:
    print(f"{t.role.upper()}: {t.content}")

## 7. Now evaluate the simulated conversation

The exact same conversational metrics from notebook 3 — nothing new to learn, just a dataset you
didn't have to write by hand.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig
from deepeval.metrics import ConversationCompletenessMetric, RoleAdherenceMetric

metrics = [RoleAdherenceMetric(model=judge), ConversationCompletenessMetric(model=judge)]

results = evaluate(
    test_cases=[simulated],
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=1),
    display_config=DisplayConfig(print_results=False),
    error_config=ErrorConfig(ignore_errors=True),
)

for tr in results.test_results:
    for md_ in tr.metrics_data:
        verdict = "PASS" if md_.success else "FAIL"
        score = f"{md_.score:.2f}" if md_.score is not None else "ERRORED"
        print(f"[{verdict}] {md_.name}: {score} -- {md_.reason or md_.error}")

## Takeaway

The full loop, no hand-written chat logs anywhere: **synthesize a scenario** (`Synthesizer`) →
**simulate a conversation** against your real chatbot (`ConversationSimulator`) → **evaluate it**
(the same metrics from notebook 3). This is how you build a multi-turn eval suite that scales past
the handful of conversations you can think to write yourself — and it's the last piece before
wiring evals into CI, covered next in the CI/CD guide.